## GX_07 — Graph analysis: do similar customers leave together?

**Question:** Do similar customers also leave together, and which groups should the retention team monitor? A shorter second part explores suspicious patterns in the transaction network.

**Notebook overview:**
- Part A — Customer similarity network: Customers are nodes, linked by similarity in age, balance, number of products, and activity—the same four features used in GX_06. GraphFrames identifies isolated customers, connected groups, and communities, then examines their churn patterns. Findings from the training data are checked on the 1,499 validation customers. Test customers are not used.
- Part B — Transaction network: Accounts are nodes and transfers are links. We look for hubs, rings, and accounts that may act as intermediaries.
Where it runs

The notebook reads the same raw CSV files from data/raw and reuses the src/ functions from GX_01–GX_02 and the segmentation code from GX_06. No manual data transfer is needed. It checks record counts against the Databricks run and stops if they differ.

**How to run it**
- **Google Colab (easiest, nothing to install):** [open this notebook in Colab](https://colab.research.google.com/github/heyydaniyal/spark-databricks-analytics/blob/main/notebooks/GX_07_graph_analysis.ipynb), then *Runtime → Run all*. The first cell installs Java and the libraries and downloads the project from GitHub (about 3–5 minutes). At the end, *File → Download → .ipynb* gives the executed notebook.
- **Own computer:** Java 17 or 21, then `pip install pyspark==4.1.2 graphframes-py==0.12.3 pandas matplotlib pyarrow`. On Windows Spark also needs `winutils.exe` and `hadoop.dll` with the `HADOOP_HOME` variable set.

We use pyspark 4.1 because the GraphFrames 0.12.3 library is built for Spark 4.1. Databricks runs Spark 4.2; the checks in Steps 1–2 show that the customers and segments come out identical. When Spark starts, it downloads the GraphFrames library from Maven, so it needs internet.

### 0. Prepare the environment and start Spark

**First cell:** on Google Colab it installs Java 17 and the two libraries and downloads the project from GitHub. On your own computer it only checks the setup.

**Second cell:** start a local Spark session with the GraphFrames library and import the shared helpers. The look-alike search and the network inputs live in `src/graph.py`; the segment code is reused from `src/segments.py`.

**No files are written.** Everything stays in memory, and `connectedComponents` uses its GraphX version, which needs no checkpoint folder on disk.

In [0]:
# Prepares the machine. On Google Colab it installs everything and downloads the project;
# on your own computer it only checks the setup. Safe to run more than once.
import os
import subprocess
import sys

REPO_URL = "https://github.com/heyydaniyal/spark-databricks-analytics.git"
COLAB_REPO = "/content/spark-databricks-analytics"
IN_COLAB = "COLAB_RELEASE_TAG" in os.environ or "google.colab" in sys.modules

if IN_COLAB:
    # 1. Java 17: Spark 4 needs Java 17 or newer.
    subprocess.run("apt-get -qq update > /dev/null && apt-get -qq install -y openjdk-17-jdk-headless > /dev/null",
                   shell=True, check=True)
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    # 2. The same library versions as a local run.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyspark==4.1.2", "graphframes-py==0.12.3"],
                   check=True)
    # 3. The project: the src/ helpers and the two raw CSV files in data/raw.
    if not os.path.isdir(COLAB_REPO):
        subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL, COLAB_REPO], check=True)
    os.chdir(os.path.join(COLAB_REPO, "notebooks"))   # the notebook finds src/ and data/ one folder up
    print("Colab ready: Java 17, pyspark 4.1.2, graphframes-py 0.12.3, project downloaded.")
elif os.name == "nt" and not os.environ.get("HADOOP_HOME"):
    raise RuntimeError(
        "Windows: Spark needs winutils.exe + hadoop.dll and the HADOOP_HOME variable to load the GraphFrames "
        "library. Set them up (see the setup notes above) or run this notebook on Google Colab instead."
    )
else:
    print("Local run: using the Python environment of this notebook.")

if not os.path.isfile(os.path.join("..", "src", "graph.py")):
    raise FileNotFoundError("src/graph.py not found: the Phase 6 files must be in the repo (and pushed, for Colab).")


In [0]:
# Make the shared helpers in src/ importable (this notebook lives in notebooks/).
import os
import sys

REPO_ROOT = os.path.abspath("..")
if REPO_ROOT not in sys.path:
    sys.path.append(REPO_ROOT)

from math import comb

import matplotlib.pyplot as plt
import pandas as pd
import pyspark
from IPython.display import display
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F

from src.config import (
    EXPECTED_ROWS, GRAPHFRAMES_PACKAGE, LOCAL_CHURN_CSV, LOCAL_TRANSACTIONS_CSV, SEED, SPLIT_RATIOS,
)

# Local Spark with the GraphFrames library. The first start downloads the library from Maven
# (internet needed once); later starts use the local copy.
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("GX_07_graph_analysis")
    .config("spark.jars.packages", GRAPHFRAMES_PACKAGE)
    .config("spark.sql.shuffle.partitions", "8")   # small data: 8 tasks per shuffle instead of 200
    .config("spark.ui.showConsoleProgress", "false")   # no progress bars in the saved outputs
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

import graphframes
from graphframes import GraphFrame
from graphframes.lib import AggregateMessages as AM

from src.cleaning import clean_churn, clean_transactions
from src.features import build_gold_features
from src.graph import (
    LOOKALIKE_K, LOOKALIKE_MAX_DISTANCE, churn_band, degree_band, exact_lookalikes, fit_lookalike_lsh,
    most_common_label, nearest_lookalikes, transaction_graph_inputs, undirected_links,
)
from src.ingest import read_raw_csv
from src.ml import KEY_COL, LABEL_COL, SPLIT_COL, add_split_column
from src.schemas import CHURN_SCHEMA, TRANSACTIONS_SCHEMA
from src.segments import (
    CLUSTER_COL, SEGMENT_ID_COL, SEGMENT_NAME_COL, SEGMENT_VECTOR_COL, add_segment_labels,
    build_segment_prep_stages, fit_best_kmeans, fixed_layout, segment_lookup, segment_profile,
)

# Same segmentation settings as GX_06 (k = 5, best of seeds 1-10)
SEGMENT_K = 5
SEGMENT_SEEDS = list(range(1, 11))
LPA_ROUNDS = 20          # label propagation rounds (Step 9 explains the choice)
MIN_COMMUNITY = 20       # communities smaller than this are too small to judge on their own

# Same colours as GX_03 and GX_06, so the slides look like one deck
COLOR_MAIN = "#27ae60"
COLOR_DARK = "#0b5345"
COLOR_GREY = "#7f8c8d"

print("Spark:", spark.version, "| GraphFrames (Python):", graphframes.__version__)
if not spark.version.startswith("4.1"):
    print("WARNING: GraphFrames 0.12.3 is built for Spark 4.1; install pyspark 4.1.x for this notebook.")


### 1. Rebuild the customers and the transactions

**Approach:** Read the two raw files and run the same bronze → silver → gold steps as GX_01 and GX_02, including the fixed train / validation / test split.

**Why:** The graph must describe exactly the same customers as the rest of the project. The cell stops with an error if the split differs from the one GX_02 produced on Databricks.

In [0]:
# Bronze -> silver -> gold, exactly as GX_01 and GX_02 do it (same functions, same options).
bronze_churn = read_raw_csv(spark, LOCAL_CHURN_CSV, CHURN_SCHEMA, encoding="ISO-8859-1")
gold = (
    build_gold_features(clean_churn(bronze_churn))
    .transform(lambda df: add_split_column(df, SPLIT_RATIOS, SEED))
    .cache()
)

bronze_transactions = read_raw_csv(spark, LOCAL_TRANSACTIONS_CSV, TRANSACTIONS_SCHEMA, quote='"', escape='"')
transactions = clean_transactions(bronze_transactions).cache()

split_counts = (
    gold.groupBy(SPLIT_COL)
    .agg(F.count("*").alias("customers"), F.sum(LABEL_COL).alias("leavers"))
    .orderBy(F.col("customers").desc())
)
display(split_counts.toPandas())

# The numbers GX_02 produced on Databricks. If any differs, stop: we would not be analysing
# the same customers as the rest of the project.
DATABRICKS_SPLIT = {"train": (7_000, 1_426), "val": (1_499, 305), "test": (1_501, 306)}
local_split = {r[SPLIT_COL]: (r["customers"], r["leavers"]) for r in split_counts.collect()}
assert local_split == DATABRICKS_SPLIT, f"Split differs from Databricks: {local_split}"
assert transactions.count() == EXPECTED_ROWS["transactions"]
print("Same customers, same split, same transactions as on Databricks.")


**Result:** 000 / 1499 / 1501 customers with 1426 / 305 / 306 leavers, identical to GX_02 on Databricks, plus the 1000 transactions. The split comes out the same on a laptop because it is built from a hash of the customer number, not from a random draw.

### 2. Rebuild the five segments

**Approach:** The GX_06 segmentation, with the same code and settings.

**Why:** First, we want to know whether the network finds anything the five segments do not. Then, the segmentation gives each customer the scaled four-number description that the look-alike search in Step 3 uses.

In [0]:
# GX_06's segmentation, rebuilt with the same code: fixed layout, scaling learned on train,
# best of 10 KMeans starts, names from the profile rule.
train = fixed_layout(gold.filter(F.col(SPLIT_COL) == "train")).cache()
val = gold.filter(F.col(SPLIT_COL) == "val").cache()

prep_model = Pipeline(stages=build_segment_prep_stages()).fit(train)
kmeans_model, _ = fit_best_kmeans(prep_model.transform(train), SEGMENT_K, SEGMENT_SEEDS)
segment_model = PipelineModel(stages=prep_model.stages + [kmeans_model])
lookup = segment_lookup(segment_profile(segment_model.transform(train), CLUSTER_COL).collect())

# Both tables keep the scaled 4-number description of each customer (segment_features):
# the look-alike search in Step 3 uses exactly these numbers.
train_seg = add_segment_labels(segment_model.transform(train), lookup).cache()
val_seg = add_segment_labels(segment_model.transform(val), lookup).cache()

segment_sizes = (
    train_seg.groupBy(SEGMENT_ID_COL, SEGMENT_NAME_COL)
    .agg(F.count("*").alias("customers"), F.round(F.avg(LABEL_COL) * 100, 1).alias("churn_pct"))
    .orderBy(SEGMENT_ID_COL)
)
display(segment_sizes.toPandas())

DATABRICKS_SEGMENTS = {1: 1_259, 2: 1_764, 3: 1_462, 4: 1_807, 5: 708}   # GX_06 on Databricks
local_segments = {r[SEGMENT_ID_COL]: r["customers"] for r in segment_sizes.collect()}
assert local_segments == DATABRICKS_SEGMENTS, f"Segments differ from GX_06: {local_segments}"
print("Same five segments as GX_06 on Databricks.")


**Result:** The same five segments with the same sizes and churn rates as GX_06 on Databricks (1259 / 1764 / 1462 / 1807 / 708 customers; churn 10.5% to 37.4%). This works because GX_06 fixes the row order and keeps the best of 10 KMeans starts, so the result does not depend on the machine.

### 3. Finding similar customers with LSH

**Part A — The customer similarity network**
How we measure similarity. Each customer is represented by four standardized features: age, balance, number of products, and activity. Standardization uses the means and standard deviations learned from the training customers. Similarity is measured by Euclidean distance between these four values. Churn is never used to create links.

**The linking rule:** Each customer is linked to up to 10 nearest neighbours within a distance of 0.5.
- A distance of 0.5 corresponds to roughly 5 years of age or €31,000 in balance, if all other features are identical. Differences across features contribute jointly to the distance.
- A one-product difference contributes about 1.7, and a difference in activity contributes about 2.0. Both exceed the threshold, so linked customers must have the same product count and activity status.
- The limit of 10 neighbours keeps each neighbourhood small. With all 10 present, each neighbour contributes 10 percentage points to its churn rate; with fewer neighbours, each contributes more.
Why use LSH? Comparing every pair would require about 24.5 million comparisons for 7,000 customers, or 500 billion for one million. MLlib’s BucketedRandomProjectionLSH uses three random projections to place customers into buckets. approxSimilarityJoin compares customers sharing a bucket, distributing the work across Spark.

This is an approximate search, so it may miss some nearby customers. A fixed seed and customer-ID tie-breaking help make the resulting network reproducible.

In [0]:
# LSH: put customers into buckets of nearby customers, then compare only customers that share a bucket.
lsh_model = fit_lookalike_lsh(train_seg, SEGMENT_VECTOR_COL, SEED)
train_pairs = nearest_lookalikes(lsh_model, train_seg, train_seg).cache()

n_train = train_seg.count()
per_customer = train_pairs.groupBy("src").count()
found_counts = (
    train_seg.select(F.col(KEY_COL).alias("src"))
    .join(per_customer, "src", "left")
    .fillna(0, ["count"])
    .select(
        F.sum((F.col("count") == LOOKALIKE_K).cast("int")).alias("with_10"),
        F.sum(((F.col("count") > 0) & (F.col("count") < LOOKALIKE_K)).cast("int")).alias("with_1_to_9"),
        F.sum((F.col("count") == 0).cast("int")).alias("with_none"),
    )
    .first()
)
all_pairs = n_train * (n_train - 1) // 2
print(f"Customer pairs a full comparison would check: {all_pairs:,}")
print(f"Look-alike pairs kept (customer -> one of its {LOOKALIKE_K} closest within {LOOKALIKE_MAX_DISTANCE}): {train_pairs.count():,}")
print(f"Customers with {LOOKALIKE_K} look-alikes: {found_counts['with_10']:,} | with 1-9: {found_counts['with_1_to_9']:,} "
      f"| with none: {found_counts['with_none']:,}")
display(train_pairs.select("distance").summary("min", "25%", "50%", "75%", "max").toPandas())


**Result:** We have, 68554 look-alike pairs, 6737 customers have their full 10 look-alikes, 235 have between 1 and 9 and 28 have none within 0.5. 
At least a quarter of the pairs have distance exactly 0, meaning, many customers are identical on all four numbers (no balance, same age, same products and activity). Customers with fewer than 10 look-alikes have an unusual profile. 

### 4. Does LSH miss look-alikes?

**Approach:** For 500 training customers, find the true 10 closest look-alikes by comparing them with all 7000 customers (the exact way), and count how many of them LSH also found.

**Why:** LSH is approximate. Before building on it, we measure what it misses.

In [0]:
# 500 training customers, chosen by a hash of their number (the same 500 on every run).
sample = train_seg.orderBy(F.xxhash64(F.col(KEY_COL).cast("string"), F.lit(SEED))).limit(500).cache()

exact_pairs = exact_lookalikes(sample, train_seg, SEGMENT_VECTOR_COL).cache()
lsh_pairs_for_sample = train_pairs.join(sample.select(F.col(KEY_COL).alias("src")), "src")

n_exact = exact_pairs.count()
n_found = exact_pairs.join(lsh_pairs_for_sample, ["src", "dst"], "left_semi").count()
print(f"Exact search compared {sample.count() * n_train:,} pairs for these 500 customers.")
print(f"True look-alikes (exact search): {n_exact:,} | also found by LSH: {n_found:,} ({n_found / n_exact:.2%})")


**Result:** LSH found 4926 of the 4927 true look-alikes (99.98%). The exact way needed 3.5 million comparisons for only 500 customers. At the bank's real size LSH is what makes a look-alike network possible, and this check shows it costs almost nothing in accuracy.

### 5. Build the graph, then ask: Who has few look-alikes?

**Approach:** A GraphFrame needs two DataFrames. 
Vertices: one row per customer, with an `id`.
Edges: one row per link, with `src` and `dst`. A customer is linked to another if either one picked the other as a look-alike. Each link is stored in both directions, because GraphFrames edges have a direction and "look alike" does not.

In [0]:
# Vertices: the training customers with what we want to read on the graph.
VERTEX_COLS = [LABEL_COL, SEGMENT_ID_COL, SEGMENT_NAME_COL,
               "Age", "Balance", "NumOfProducts", "IsActiveMember", "Geography", "Gender"]
vertices = train_seg.select(F.col(KEY_COL).alias("id"), *VERTEX_COLS).cache()

# Edges: one look-alike link = two directed edges (A -> B and B -> A).
edges = undirected_links(train_pairs).cache()

graph = GraphFrame(vertices, edges)
n_links = edges.count() // 2
print(f"Customers (vertices): {vertices.count():,}")
print(f"Look-alike links: {n_links:,} (stored as {edges.count():,} directed edges)")


**Result:** 7000 customers and 44071 look-alike links.

**Next steps:** count each customer's links with GraphFrames (`outDegrees`) and compare churn across customers with few and many look-alikes.

In [0]:
# outDegree = number of look-alike links of each customer (every link is stored in both
# directions, so in- and out-degree are equal). Customers without any link are not in
# outDegrees, so they are added back with 0.
links_per_customer = (
    vertices.join(graph.outDegrees.withColumnRenamed("outDegree", "links"), "id", "left")
    .fillna(0, ["links"])
    .cache()
)

display(
    links_per_customer.groupBy(degree_band("links").alias("look_alike_links"))
    .agg(
        F.count("*").alias("customers"),
        F.sum(LABEL_COL).alias("leavers"),
        F.round(F.avg(LABEL_COL) * 100, 1).alias("churn_pct"),
        F.round(F.avg("Age"), 1).alias("avg_age"),
        F.round(F.avg((F.col("NumOfProducts") >= 3).cast("int")) * 100, 1).alias("three_plus_products_pct"),
    )
    .orderBy("look_alike_links")
    .toPandas()
)
print("Links per customer:")
display(links_per_customer.select("links").summary("min", "25%", "50%", "75%", "max").toPandas())


**Result:** most customers (6120) have 10–15 look-alikes and churn at 19.5%, close to the average. The 263 customers with fewer than 10 look-alikes are very different:
- no look-alike at all: 28 customers, 71.4% left;
- 1–4 look-alikes: 102 customers, 73.5% left;
- 5–9 look-alikes: 133 customers, 56.4% left.

These customers are older, with average ages of roughly 50–58, and often hold three or more products: 64–70% in the first two groups, compared with just 1.3% in the group with 10–15 look-alikes. At the other end, the 617 customers with at least 16 look-alikes have more common profiles and the lowest churn rate (9.9%).

**For the bank:** An unusual customer profile may signal higher churn risk. The graph identifies these customers through their lack of close neighbours, without an explicit rule targeting their characteristics. This is consistent with GX_03, where customers with 3–4 products had churn rates of 83–100%.

### 6. Do look-alikes leave together?

**Approach:** GraphFrames' `aggregateMessages` lets every customer collect a message from each of its neighbours. Here the message is the neighbour's `Exited` (1 = left), and the average of the messages is the share of the customer's look-alikes who left. 
This is the building block of most graph algorithms: work is done along the links, in parallel.

**Then:** Look at every link once. Does it join two leavers, one leaver, or none? Compare that with the counts we would expect if leaving had nothing to do with looking alike (each customer leaving with the overall 20.4%). The link list comes from GraphFrames' motif search, `find("(a)-[]->(b)")`.

In [0]:
# aggregateMessages: every edge sends the neighbour's Exited (0/1) to the customer at its
# source; the average of what a customer receives = share of its look-alikes who left.
lookalike_churn = graph.aggregateMessages(
    [F.avg(AM.msg).alias("lookalikes_left_share")],
    sendToSrc=[AM.dst[LABEL_COL]],
)
with_share = vertices.join(lookalike_churn, "id").cache()

display(
    with_share.groupBy(LABEL_COL)
    .agg(
        F.count("*").alias("customers"),
        F.round(F.avg("lookalikes_left_share") * 100, 1).alias("avg_pct_of_lookalikes_who_left"),
    )
    .orderBy(LABEL_COL)
    .toPandas()
)

# Each link once (a.id < b.id): how many links join two leavers, one leaver, or none -
# compared with what we would see if leaving had nothing to do with looking alike.
links = graph.find("(a)-[]->(b)").filter("a.id < b.id")
p = train_seg.agg(F.avg(LABEL_COL)).first()[0]   # 20.4% of training customers left
link_outcomes = (
    links.select((F.col(f"a.{LABEL_COL}") + F.col(f"b.{LABEL_COL}")).alias("leavers_in_link"))
    .groupBy("leavers_in_link").count()
    .orderBy("leavers_in_link")
    .toPandas()
)
expected_share = {0: (1 - p) ** 2, 1: 2 * p * (1 - p), 2: p ** 2}
link_outcomes["expected_if_random"] = [round(link_outcomes["count"].sum() * expected_share[n]) for n in link_outcomes["leavers_in_link"]]
display(link_outcomes)


**Result:**
- For customers who left, on average 48.5% of their look-alikes also left. For customers who stayed, only 13.1% did, almost 4 times less.
- 3,579 links join two leavers, about twice the 1,829 we would expect by chance. Links between a leaver and a stayer are much rarer than chance (8,654 against 14,298).
So leaving clusters among look-alikes. 

**Note:** This is similarity, not influence. Customers do not know each other, and the data is one snapshot in time. The graph shows that customers with the same profile tend to make the same choice. It does not show that one leaver makes another one leave.

### 7. Does it hold for new customers?

**Approach:** For each validation customer, we find up to 10 similar training customers, whose churn outcomes are already known. The proportion who left becomes that customer’s risk score.
We evaluate this score using AUC-PR, the same metric used in GX_04. It measures how well the score ranks leavers above stayers: a random ranking has a baseline of roughly the churn rate (0.20), while a perfect ranking scores 1.

**Why:** Further ahead we will identify and measured the pattern using the same customers. Checking it on separate validation customers tests whether it generalizes beyond the training data.

In [0]:
# Validation customers look for their 10 look-alikes among the TRAINING customers only:
# a new customer is compared with customers whose outcome the bank already knows.
val_pairs = nearest_lookalikes(lsh_model, val_seg, train_seg).cache()
known_outcomes = vertices.select(F.col("id").alias("dst"), LABEL_COL)

val_share = (
    val_pairs.join(known_outcomes, "dst")
    .groupBy("src")
    .agg(F.avg(LABEL_COL).alias("lookalikes_left_share"), F.count("*").alias("lookalikes"))
)
segment_rate = train_seg.groupBy(SEGMENT_ID_COL).agg(F.avg(LABEL_COL).alias("segment_churn"))

# A validation customer with no look-alike within the distance gets the overall training rate.
val_scores = (
    val_seg.select(F.col(KEY_COL).alias("src"), LABEL_COL, SEGMENT_ID_COL)
    .join(segment_rate, SEGMENT_ID_COL)
    .join(val_share, "src", "left")
    .withColumn("no_lookalike", F.col("lookalikes_left_share").isNull())
    .fillna({"lookalikes_left_share": p})
    .cache()
)

pr_evaluator = BinaryClassificationEvaluator(labelCol=LABEL_COL, metricName="areaUnderPR")

def auc_pr(df, score_col):
    """AUC-PR of a 0-1 score: how well it puts the leavers at the top of the list."""
    return round(pr_evaluator.setRawPredictionCol(score_col).evaluate(df.withColumn(score_col, F.col(score_col).cast("double"))), 3)

val_leaver_rate = val_scores.agg(F.avg(LABEL_COL)).first()[0]
display(pd.DataFrame([
    {"ranking of validation customers by": "nothing (random order)", "auc_pr": round(val_leaver_rate, 3)},
    {"ranking of validation customers by": "churn rate of their segment (GX_06)", "auc_pr": auc_pr(val_scores, "segment_churn")},
    {"ranking of validation customers by": "share of their 10 look-alikes who left", "auc_pr": auc_pr(val_scores, "lookalikes_left_share")},
]))
print(f"Validation customers: {val_scores.count():,}, leavers: {val_scores.agg(F.sum(LABEL_COL)).first()[0]}, "
      f"without any look-alike: {val_scores.filter('no_lookalike').count()}")

share_band = (
    F.when(F.col("lookalikes_left_share") == 0, "1. none of them")
    .when(F.col("lookalikes_left_share") < 0.25, "2. under 25%")
    .when(F.col("lookalikes_left_share") < 0.5, "3. 25-50%")
    .otherwise("4. 50% or more")
)
display(
    val_scores.groupBy(share_band.alias("lookalikes_who_left"))
    .agg(F.count("*").alias("val_customers"), F.sum(LABEL_COL).alias("val_leavers"),
         F.round(F.avg(LABEL_COL) * 100, 1).alias("val_churn_pct"))
    .orderBy("lookalikes_who_left")
    .toPandas()
)



**Result:**
- The neighbour-based score achieves AUC-PR of 0.608, compared with 0.203 for a random ranking and 0.289 when scoring customers by their segment’s churn rate.
- Among 540 validation customers with no leavers among their neighbours, 5.4% left. Among 220 customers whose neighbours were at least half leavers, 64.1% left.
- Two validation customers had no neighbour within the distance threshold and received the overall training churn rate as their score.
The GX_04 churn model performs better on the same customers (AUC-PR around 0.67–0.69), so it remains the tool for prioritizing contacts. The network adds an intuitive explanation: “7 of the 10 most similar training customers left.” This describes a pattern among similar customers; it does not imply that they influenced one another.

### 8. Islands: where the network separates
**Approach:** `connectedComponent` identifies groups connected by a chain of links, with no links to customers outside the group.

**Why:** These “islands” reveal where the network separates under our similarity rule. Their boundaries reflect both customer differences and the distance threshold and neighbour limit used to build the graph.

In [0]:
components = graph.connectedComponents(algorithm="graphx").cache()

island_profile = (
    segment_profile(components, "component")
    .select("component", "customers", "churn_pct", "avg_age", "zero_balance_pct",
            "one_product_pct", "two_products_pct", "three_plus_pct", "active_pct")
    .orderBy(F.col("customers").desc(), F.col("churn_pct").desc())
    .toPandas()
)
n_islands = len(island_profile)
small = island_profile[island_profile["customers"] < 50]
print(f"Islands: {n_islands} | four largest hold {island_profile['customers'].head(4).sum():,} customers "
      f"| islands under 50 customers: {len(small)} ({small['customers'].sum():,} customers)")
display(island_profile.head(10))


**Result:** Of overall 125 islands, the four largest hold 4585 customers (65%), and each one is a single combination of products and activity:

| island | customers | churn |
|---|---|---|
| 1 product, inactive | 1,473 | **33.9%** |
| 1 product, active | 1,619 | 19.5% |
| 2 products, inactive | 746 | 15.7% |
| 2 products, active | 747 | 9.1% |

This follows from the rule in Step 3: a difference in product count or activity status always exceeds the 0.5 distance threshold, so these customers cannot be linked.

The remaining 121 islands contain 2112 customers, with 116 islands having fewer than 50 customers. They are separated from the larger groups by gaps in age or balance under the linking rule. Some have strikingly different churn rates:
- 65 inactive customers with 3 products: 96.9% churn.
- 65 inactive, one-product customers, aged about 55 with no balance: 98.5% churn.
- 65 active, two-product customers, aged about 70 with no balance: 1.5% churn.

**For the bank:** Product count and activity create firm boundaries in this network because of how similarity is defined. The churn differences across islands suggest groups worth investigating, but they do not show that adding a product or increasing activity would itself reduce churn.

In [0]:
def island_name(row):
    """Short label from the island's profile: products held + activity."""
    products = ("1 product" if row.one_product_pct >= 90 else "2 products" if row.two_products_pct >= 90
                else "3+ products" if row.three_plus_pct >= 90 else "mixed products")
    activity = "active" if row.active_pct >= 90 else "inactive" if row.active_pct <= 10 else "mixed activity"
    return f"{products}, {activity} ({row.customers:,})"

top = island_profile.head(7).copy()   # the 4 large islands + the 3 islands of 65 customers
top["label"] = [island_name(r) for r in top.itertuples()]
top = top.iloc[::-1]   # largest at the top of the chart

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(top["label"], top["churn_pct"], color=COLOR_MAIN)
for y, churn in enumerate(top["churn_pct"]):
    ax.text(churn + 1, y, f"{churn:.1f}%", va="center", fontsize=9)
ax.axvline(p * 100, color=COLOR_GREY, linestyle="--", linewidth=1, label=f"All training customers ({p * 100:.1f}%)")
ax.set_xlim(0, 110)
ax.set_xlabel("Churn rate (%)")
ax.set_title("The seven largest look-alike islands: how many leave")
ax.legend(loc="lower right", fontsize=9)
plt.tight_layout()
plt.show()


### 9. Communities inside the islands

**Approach:** `labelPropagation` finds communities: groups of customers more linked to each other than to the rest of their island.
Every customer starts with its own number as a label. In each round every customer takes the label that is most common among its look-alikes (ties go to the smallest label). Groups end up sharing one label.

**Why 20 rounds:** After about 14 rounds the labels stop changing, except for a few customers who swap back and forth between two neighbouring communities every round. This is a known habit of this algorithm. The cell runs one round less as a check and counts how many customers still move.

**Why:** Segments are five broad groups. Communities are small and specific, which is the size a retention team can actually work with.

In [0]:
# Label propagation (GraphX version, the GraphFrames default): every customer starts with its
# own number as label; each round it takes the label most common among its look-alikes
# (ties -> smallest label). Run for LPA_ROUNDS rounds and, as a check, for one round less.
communities = graph.labelPropagation(maxIter=LPA_ROUNDS).cache()
one_round_less = graph.labelPropagation(maxIter=LPA_ROUNDS - 1)

still_moving = (
    communities.join(one_round_less.select("id", F.col("label").alias("label_before")), "id")
    .filter(F.col("label") != F.col("label_before"))
    .count()
)
community_profile = segment_profile(communities, "label").cache()
sizes = community_profile.select("customers").toPandas()["customers"]

print(f"Communities: {len(sizes)} | customers still changing community in the last round: {still_moving}")
print(f"Community size: median {sizes.median():.0f}, largest {sizes.max()}, "
      f"{(sizes >= MIN_COMMUNITY).sum()} communities with {MIN_COMMUNITY}+ customers "
      f"({sizes[sizes >= MIN_COMMUNITY].sum():,} customers)")

# How "pure" are the communities: share of members in the community's most common segment.
purity = (
    communities.groupBy("label", SEGMENT_NAME_COL).count()
    .withColumn("share", F.col("count") / F.sum("count").over(Window.partitionBy("label")))
    .groupBy("label").agg(F.max("share").alias("top_segment_share"))
)
print(f"Communities with at least 90% of members from one segment: "
      f"{purity.filter('top_segment_share >= 0.9').count() / len(sizes):.1%}")

# Inside each segment: the lowest and highest churn among its communities of 20+ customers.
main_segment = (
    communities.groupBy("label", SEGMENT_NAME_COL).count()
    .withColumn("rank", F.row_number().over(Window.partitionBy("label").orderBy(F.col("count").desc(), SEGMENT_NAME_COL)))
    .filter("rank = 1").select("label", F.col(SEGMENT_NAME_COL).alias("main_segment"))
)
display(
    community_profile.filter(F.col("customers") >= MIN_COMMUNITY)
    .join(main_segment, "label")
    .groupBy("main_segment")
    .agg(F.count("*").alias("communities"), F.sum("customers").alias("customers"),
         F.min("churn_pct").alias("lowest_churn_pct"), F.max("churn_pct").alias("highest_churn_pct"))
    .join(segment_sizes.select(F.col(SEGMENT_NAME_COL).alias("main_segment"),
                               F.col("churn_pct").alias("segment_churn_pct"), SEGMENT_ID_COL), "main_segment")
    .orderBy(SEGMENT_ID_COL)
    .drop(SEGMENT_ID_COL)
    .toPandas()
)


**Result:** 426 communities, typically 14 customers (largest 58). Only 58 customers (under 1%) still swap community in the last round, which does not change the picture.

- 93.7% of communities draw at least 90% of their members from one segment. Communities are finer pieces of the segments, not a different way of cutting the customers.
- Inside each segment, churn differs enormously between communities. Inactive savers (27.8% on average) contain communities from 0% to 90.9%; inactive customers with little or no savings (21.2%) from 0% to 96.0%. Even active savers (12.3%) contain one community where 51.9% left.

So the five segments are right on average, but each one hides pockets where most customers leave.

### 10. The hot communities

**Approach:** List communities with at least 20 customers and a churn rate of 50% or higher. We exclude smaller communities because their churn rates are more sensitive to individual outcomes.

In [0]:
# "Hot" communities: 20+ customers and at least half of them left (training customers).
hot = (
    community_profile.filter((F.col("customers") >= MIN_COMMUNITY) & (F.col("churn_pct") >= 50))
    .join(main_segment, "label")
    .select("label", "main_segment", "customers", "leavers", "churn_pct", "avg_age", "avg_balance",
            "zero_balance_pct", "avg_products", "active_pct", "germany_pct")
    .orderBy(F.col("churn_pct").desc(), F.col("customers").desc(), "label")
    .toPandas()
)
print(f"Hot communities: {len(hot)}, {hot['customers'].sum()} customers, {hot['leavers'].sum()} leavers "
      f"({hot['leavers'].sum() / hot['customers'].sum():.1%})")
display(hot)


**Result:** We identified 13 high-churn communities, containing 351 customers, of whom 240 left (68.4%).
- Ten communities are inactive: five groups of one-product savers aged 45–55 with average balances of €80,000–€180,000; two groups with three products; two groups of two-product savers aged 50–59; and one group of one-product customers around age 47 with no balance.
- Three communities are active: two groups of one-product savers aged 46–50 and one group of customers in their early 50s with no balance. Their churn rates range from 51.9% to 62.5%. The active-saver communities offer a more detailed view of the segment identified as the churn model’s blind spot in GX_06.
- German customers make up at least half of nine communities, compared with 25% of all training customers.

These churn rates describe the same customers used to form the communities. We therefore check whether the patterns also appear in validation data.

### 11. Do the communities hold on new customers?
**Approach:** Assign each validation customer to the most common community among their up to 10 similar training customers. Use that community’s training churn rate as the customer’s risk score. The validation customer’s own Exited value is used only to evaluate the result.

In [0]:
# Validation customers join the community most common among their 10 training look-alikes,
# and inherit that community's TRAINING churn rate. Their own Exited is only read to check.
val_community = most_common_label(val_pairs, communities.select("id", "label"), "label")
community_rate = communities.groupBy("label").agg(
    F.avg(LABEL_COL).alias("community_churn"), F.count("*").alias("community_size")
)
val_scores = (
    val_scores.join(val_community, "src", "left")
    .join(community_rate, "label", "left")
    .fillna({"community_churn": p, "community_size": 0})
    .withColumn("in_hot_community", (F.col("community_size") >= MIN_COMMUNITY) & (F.col("community_churn") >= 0.5))
    .cache()
)

by_band = (
    val_scores.groupBy(churn_band("community_churn").alias("community_churn_on_train"))
    .agg(F.count("*").alias("val_customers"), F.sum(LABEL_COL).alias("val_leavers"),
         F.round(F.avg(LABEL_COL) * 100, 1).alias("val_churn_pct"))
    .orderBy("community_churn_on_train")
    .toPandas()
)
display(by_band)
display(
    val_scores.groupBy("in_hot_community")
    .agg(F.count("*").alias("val_customers"), F.sum(LABEL_COL).alias("val_leavers"),
         F.round(F.avg(LABEL_COL) * 100, 1).alias("val_churn_pct"))
    .orderBy("in_hot_community")
    .toPandas()
)
print("AUC-PR, ranking validation customers by their community's churn rate:", auc_pr(val_scores, "community_churn"))


**Result:** The community patterns also hold on validation.
- In communities with training churn below 10%, validation churn is 7.2% across 725 customers. In communities with training churn of at least 50%, it is 67.3% across 205 customers.
- The 77 validation customers assigned to the 13 hot communities have a churn rate of 66.2%, compared with 17.9% among everyone else.
- Using each community’s training churn rate as a score gives AUC-PR of 0.624, slightly above the neighbour-based score in Step 7 (0.608).

These results suggest that the communities capture churn patterns that generalize beyond the training data. The highest-risk band contains 205 customers, rather than 77, because it also includes communities with fewer than 20 training customers, which were excluded from the hot-community list.

In [0]:
fig, ax = plt.subplots(figsize=(10, 4.5))
labels = [band.split(". ")[1] for band in by_band["community_churn_on_train"]]
bars = ax.bar(labels, by_band["val_churn_pct"], color=COLOR_MAIN)
for bar, churn, n in zip(bars, by_band["val_churn_pct"], by_band["val_customers"]):
    ax.text(bar.get_x() + bar.get_width() / 2, churn + 1.5, f"{churn:.1f}%\n({n} customers)", ha="center", fontsize=9)
ax.axhline(val_leaver_rate * 100, color=COLOR_GREY, linestyle="--", linewidth=1,
           label=f"All validation customers ({val_leaver_rate * 100:.1f}%)")
ax.set_ylim(0, 100)
ax.set_xlabel("Churn rate of the customer's look-alike community (learned on training customers)")
ax.set_ylabel("Validation customers who left (%)")
ax.set_title("New customers leave like their look-alike community")
ax.legend(loc="upper left", fontsize=9)
plt.tight_layout()
plt.show()


### 12. Triangles of leavers 

**Approach:** GraphFrames's `find` searches for a pattern in the network. Here the pattern is a triangle: three customers who are all look-alikes of each other. We count how many triangles have 0, 1, 2 or 3 leavers and compare with chance.

**Why:** A triangle is the smallest tight group. If leavers come in tight groups, all-leaver triangles should be far more common than chance.

In [0]:
# Motif: three customers who are all look-alikes of each other (a triangle).
# Each triangle is found once per direction, so we keep the one with a.id < b.id < c.id.
triangles = (
    graph.find("(a)-[]->(b); (b)-[]->(c); (c)-[]->(a)")
    .filter("a.id < b.id AND b.id < c.id")
    .select(
        (F.col(f"a.{LABEL_COL}") + F.col(f"b.{LABEL_COL}") + F.col(f"c.{LABEL_COL}")).alias("leavers_in_triangle"),
        F.col(f"a.{SEGMENT_NAME_COL}").alias("segment_of_a"),
    )
    .cache()
)
n_triangles = triangles.count()

# If leaving had nothing to do with looking alike, each member would leave with probability p.
triangle_outcomes = triangles.groupBy("leavers_in_triangle").count().orderBy("leavers_in_triangle").toPandas()
triangle_outcomes["expected_if_random"] = [
    round(n_triangles * comb(3, n) * p ** n * (1 - p) ** (3 - n)) for n in triangle_outcomes["leavers_in_triangle"]
]
print(f"Triangles of look-alikes: {n_triangles:,}")
display(triangle_outcomes)

# Where the all-leaver triangles sit (segment of the first member; a triangle almost never spans two segments).
display(
    triangles.filter("leavers_in_triangle = 3").groupBy("segment_of_a").count()
    .orderBy(F.col("count").desc()).toPandas()
)


**Result:** The network contains 123,137 triangles. Of these, 5,448 contain three leavers, compared with 1,041 expected by chance—about five times as many. Triangles with no leavers are also more common than expected (86,547 vs 62,172), while mixed triangles are less common. Closely connected, similar customers tend to share the same churn outcome.

Most all-leaver triangles occur in the inactive and older segments (5,018 combined), compared with 430 in the two active segments.

### Part B — The transaction network
**Approach:** `bank_transactions.cs` contains 1,000 transfers recorded between 10:01 and 11:00 on 17 January 2025. Its account IDs do not match the customer IDs (GX_01), so the transfers cannot be linked to the churn data. We represent accounts as nodes and transfers as directed links.

**Question:** Does the network contain hubs, transfer cycles, or accounts that receive and pass on money? These patterns can help identify activity worth investigating, but do not establish fraud on their own.

In [0]:
tx_vertices, tx_edges = transaction_graph_inputs(
    transactions, ["transaction_id", "amount", "transaction_type", "event_time", "transaction_status", "fraud_flag"]
)
tx_graph = GraphFrame(tx_vertices.cache(), tx_edges.cache())

print(f"Accounts: {tx_graph.vertices.count():,} | transactions (edges): {tx_graph.edges.count():,}")
display(
    tx_graph.vertices.groupBy("transactions").agg(F.count("*").alias("accounts"))
    .orderBy("transactions").toPandas()
)

tx_components = tx_graph.connectedComponents(algorithm="graphx")
print(f"Separate pieces (connected components): {tx_components.select('component').distinct().count()}")
display(
    tx_components.groupBy("component").agg(F.count("*").alias("accounts"))
    .groupBy("accounts").agg(F.count("*").alias("components"))
    .orderBy("accounts").toPandas()
)


**Result:** The network contains 1971 accounts and 1000 transfers. Most accounts appear only once: 1943 appear once, 27 twice, and one three times.

`connectedComponents` identifies 971 separate groups: 945 pairs, 23 groups of three, and three groups of four. There are no hubs or large connected groups.

Next, we use motif searches to find two transfer patterns: money passing through an account (a → b → c) and transfers in both directions (a → b → a). We then compare the share of transfers flagged as fraud between those involving repeat accounts and all other transfers.

In [0]:
# Pass-through: money arrives at an account and leaves it again (a -> b -> c).
chains = tx_graph.find("(a)-[e1]->(b); (b)-[e2]->(c)")
n_chains = chains.count()
n_ordered = chains.filter("e1.event_time <= e2.event_time").count()
# Round trip: money goes a -> b and comes back b -> a.
n_round_trips = tx_graph.find("(a)-[e1]->(b); (b)-[e2]->(a)").count()
print(f"Pass-through chains a -> b -> c: {n_chains} (money in before money out: {n_ordered}) | round trips: {n_round_trips}")

# Fraud flag: transactions that involve an account seen more than once vs all others.
repeat_accounts = tx_graph.vertices.filter("transactions > 1").select("id")
flagged = (
    tx_graph.edges
    .join(repeat_accounts.withColumnRenamed("id", "src").withColumn("src_repeat", F.lit(True)), "src", "left")
    .join(repeat_accounts.withColumnRenamed("id", "dst").withColumn("dst_repeat", F.lit(True)), "dst", "left")
    .withColumn("touches_repeat_account", F.coalesce("src_repeat", F.lit(False)) | F.coalesce("dst_repeat", F.lit(False)))
)
display(
    flagged.groupBy("touches_repeat_account")
    .agg(F.count("*").alias("transactions"), F.round(F.avg(F.col("fraud_flag").cast("int")) * 100, 1).alias("fraud_flag_pct"))
    .orderBy("touches_repeat_account").toPandas()
)


**Result:** We found 18 pass-through chains, including 12 where the incoming transfer precedes the outgoing one, and no round trips.

Among the 55 transfers involving repeat accounts, 58.2% are flagged as fraud, compared with 47.5% of other transfers. This small sample leaves substantial uncertainty—roughly ±13 percentage points around the repeat-account rate—so the observed difference is not convincing evidence of a relationship.

**Conclusion:** This one-hour sample provides too little connected activity to draw firm conclusions about transaction patterns. The fraud flag is also close to evenly split, as noted in GX_01. The analysis provides a starting point for a longer transaction history, where persistent hubs, cycles, and pass-through activity could be investigated. These results do not establish a link between network structure and fraud.

### 13. Summary

**Part A — Customer similarity network**
We built a network of 7000 training customers using MLlib LSH, producing 44071 links, and checked the findings on 1499 validation customers.
- Similar customers share churn patterns. Triangles containing three leavers are five times more common than expected by chance. This shows similarity, not influence.
- The pattern holds on validation: churn is 64.1% when at least half of a customer’s neighbours left, compared with 5.4% when none left.
- The network adds detail beyond the segments. It identifies 13 high-churn communities, containing 351 training customers with 68.4% churn. Validation customers assigned to these communities have 66.2% churn.
- The churn model remains better for prioritizing contacts: its AUC-PR is around 0.67–0.69, compared with 0.608 for the neighbour-based score. The network adds an understandable explanation and highlights smaller groups.

**Part B — Transaction network**
The one-hour sample contains 1971 accounts, mostly in isolated pairs. We found 18 pass-through chains, no round trips, and no convincing relationship between repeat-account activity and fraud flags. A longer transaction history is needed for meaningful conclusions.

**Next: GX_08 — Streaming on Databricks**
Spark Structured Streaming processes new customer files, applies the saved churn model, and updates a table of high-risk alerts by country. The similarity network remains a separate analysis and is not rebuilt for each batch.